# Module 6: Quantization with LLM Compressor

In Module 5 you tuned the engine flags and raised throughput by raising `--max-num-seqs`, but tuning config only goes as far as the model on the card lets it. When the KV cache is your limit and you have set the flags as far as they will go, the next move is to make the model itself smaller. This module shrinks the weights with quantization. You convert your [vLLM](https://docs.vllm.ai) model to FP8 with [LLM Compressor](https://github.com/vllm-project/llm-compressor), measure the footprint it saved, confirm the answers still hold, and trace that saved memory straight back to the KV cache headroom from Modules 4 and 5. Smaller weights leave more room for the cache, and more cache buys more concurrency.

## Learning objectives
- Explain why fewer bits per weight frees GPU memory for the KV cache
- Quantize a model to FP8 with a single LLM Compressor `oneshot` pass
- Measure the on-disk footprint and read it as a proxy for GPU memory saved
- Run a quick quality check and confirm a simple task survives quantization
- Read the `kv_cache_usage_perc` gauge against the quantized model and see the headroom you bought
- Decide when FP8 is enough and when to reach for the heavier W4A16 scheme

## Prerequisites
- Finished Module 5, with the server tuned and the throughput knee in view
- **A machine with a CUDA GPU to run the quantize step (sections 1 to 5).** On the hosted workshop your JupyterLab pod has **no GPU**, so these cells default to a read-along. See the note in section 1. The quality and headroom checks (sections 6 to 7) run from this notebook against a live `VLLM_HOST`.
- The `llmcompressor` package (installed on the GPU machine, not the notebook pod)
- About 20 minutes

References: [LLM Compressor](https://github.com/vllm-project/llm-compressor) &middot; [LLM Compressor docs](https://docs.vllm.ai/projects/llm-compressor) &middot; [compressed-tensors format](https://github.com/neuralmagic/compressed-tensors) &middot; [vLLM quantization](https://docs.vllm.ai/en/latest/features/quantization/index.html)

## Quantization design basics

A model's weights are usually stored in 16-bit floats, BF16 or FP16, which is about 2 bytes per parameter. Quantization stores them in fewer bits, so the weights block on the card shrinks. The memory you free does not vanish: at the same `--gpu-memory-utilization`, vLLM hands it to the KV cache, and a bigger cache holds more concurrent requests before preemption.

- **FP8** keeps an 8-bit float per weight, roughly halving the weight memory versus BF16. It is the gentle option: accuracy loss is usually small, and modern GPUs run FP8 fast. **FP8_DYNAMIC** also quantizes activations at runtime with no calibration step, which makes it the easiest scheme to apply.
- **W4A16** packs weights into 4 bits while keeping activations at 16 bits. It shrinks the model far more, to about a quarter of BF16, but it needs a calibration pass over sample data and tends to cost more accuracy.

Two terms are worth separating. **Weight quantization** shrinks the stored model. **Activation quantization** shrinks the intermediate tensors computed at runtime. FP8_DYNAMIC does weights ahead of time and activations on the fly. The payoff is the same either way: you trade a little accuracy for a lot of headroom.

![A GPU memory bar where BF16 weights at about two bytes per parameter shrink to FP8 at about one byte, halving the weights block so the freed memory becomes a larger KV cache that holds more concurrent requests](images/06_quantization_architecture.png)

## 1. Setup

This step needs the quantization toolchain (`llmcompressor` + `torch` + `transformers`) and **a CUDA GPU**, because the conversion loads the full model onto the device.

> **Where this runs.** The quantize step (sections 1 to 5) runs **on a GPU**, not in this JupyterLab pod. On the hosted workshop the notebook pod is CPU-only, so the cells below default to a read-along: `RUN_QUANTIZE = False` skips the heavy install and the conversion so you can read the code without a hiccup. To run it for real, set `RUN_QUANTIZE = True` on a machine with a GPU:
> - **Path B, your own GPU machine:** a GPU box or a Colab GPU. This is the simplest way to do the conversion hands-on.
> - **Path A, hosted:** your only GPU is the one your vLLM serves on. To use it you would free it first (`kubectl scale deploy/vllm --replicas=0`) and run the conversion as a GPU Job on that node, because the notebook kernel itself cannot reach the card.
>
> Either way, the **quality and headroom checks (sections 6 to 7) run right here** against whatever `VLLM_HOST` serves. Point it at a pre-quantized endpoint to compare.

In [ ]:
# Read-along by default. Installing torch + llmcompressor only makes sense on a GPU
# machine, not the CPU notebook pod, so leave this False in the hosted workshop and
# set it True when you run sections 1 to 5 on a GPU.
RUN_QUANTIZE = False

if RUN_QUANTIZE:
    %pip install -q llmcompressor "transformers>=4.44" "torch>=2.3"
else:
    print("Read-along mode (RUN_QUANTIZE = False): skipping the GPU-only install.")

## 2. Configure endpoint and model

`get_settings()` reads your connection details from the environment, the same as every module. The source model is whatever you already serve, and the quantized copy lands in a local directory next to this notebook. Printing the source model and the output path makes the conversion target explicit before you run it.

In [ ]:
# Setup: settings and an output directory for the quantized model.
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings

settings = get_settings()
SOURCE_MODEL = settings.model_name          # the model you already serve
QUANT_DIR = os.path.join(os.getcwd(), "quantized-fp8")
print("source model :", SOURCE_MODEL)
print("output dir   :", QUANT_DIR)

**What you should see:** your source model id and the local directory the quantized copy will be written to.

## 3. Quantize to FP8

Load the model, define a one-line recipe, and run `oneshot`. FP8_DYNAMIC needs no calibration data, so this is a single pass with no dataset. The `ignore=["lm_head"]` keeps the output projection in full precision, which is standard practice because quantizing it tends to hurt quality for little memory gain.

This cell runs **on a GPU** (see section 1), not the JupyterLab pod. For a 4B model the unquantized weights are a few GB, which fits the workshop card once it is freed.

In [ ]:
# Requires a GPU and the llmcompressor package. Runs only when RUN_QUANTIZE is True.
if not RUN_QUANTIZE:
    print("Read-along mode: this is the FP8 conversion. Set RUN_QUANTIZE = True on a GPU to run it.")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from llmcompressor import oneshot
    from llmcompressor.modifiers.quantization import QuantizationModifier

    model = AutoModelForCausalLM.from_pretrained(SOURCE_MODEL, torch_dtype="auto", device_map="cuda")
    tokenizer = AutoTokenizer.from_pretrained(SOURCE_MODEL)

    # FP8 dynamic: per-tensor weight quantization plus runtime activation scaling.
    recipe = QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC", ignore=["lm_head"])

    oneshot(model=model, recipe=recipe)

    # Save in compressed-tensors format, which vLLM loads directly.
    model.save_pretrained(QUANT_DIR, save_compressed=True)
    tokenizer.save_pretrained(QUANT_DIR)
    print("saved quantized model to", QUANT_DIR)

**What you should see:** progress logs from LLM Compressor as it applies the modifier, then a confirmation that the quantized model is saved. The run takes a few minutes for a small model. If you hit an out-of-memory loading the source model, you are on too small a card for this step; use a smaller source model.

## 4. Measure the footprint on disk

The clearest first signal is size. Compare the quantized directory against what the full-precision weights would take. FP8 should land near half the BF16 size. On disk this is a proxy for what you save in GPU memory, which is the number that matters for KV cache headroom.

In [ ]:
# Report the size of the quantized model directory on disk (only if it exists).
def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1e9

if os.path.isdir(QUANT_DIR):
    print(f"quantized FP8 size on disk: {dir_size_gb(QUANT_DIR):.2f} GB")
else:
    print("No quantized model on disk yet (read-along mode).")
print("BF16 weights are about 2 bytes per parameter; FP8 is about 1,")
print("so a 4B model drops from roughly 8 GB to roughly 4 GB of weights.")

**What you should see:** a size near half what the full-precision weights would take. That freed GPU memory is what becomes extra KV cache when you serve the quantized model at the same `--gpu-memory-utilization`.

### Count the concurrency you bought

The footprint drop is the means, not the end. The end is concurrency. Every gigabyte of weights you shave becomes a gigabyte of KV cache at the same `--gpu-memory-utilization`, and a gigabyte of cache is thousands of tokens. Convert the format you picked into the requests it lets you run at once.

![A ladder from BF16 to FP8 to W4A16 halving the weight bytes at each step, with the freed memory becoming more KV cache, plus a side note that weights are safe to quantize while attention and the output head are not.](images/06_quantization_format_ladder.png)

In [ ]:
# Turn weight footprint into KV cache headroom and concurrency.
# Reuses the KV-bytes-per-token number from Module 2.
card_gb            = 20.0
util               = 0.90
kv_bytes_per_token = 147456    # Module 2: ~144 KB for a 4B GQA model
ctx                = 2048

print(f"{'format':>8} {'weights':>9} {'cache GB':>9} {'token-slots':>12} {'seqs @2048':>11}")
for fmt, weights_gb in [("BF16", 8.0), ("FP8", 4.0), ("W4A16", 2.0)]:
    cache_gb = util * card_gb - weights_gb
    slots    = cache_gb * 1e9 / kv_bytes_per_token
    print(f"{fmt:>8} {weights_gb:>6.0f} GB {cache_gb:>9.1f} {slots:>12,.0f} {slots/ctx:>11.0f}")

**What you should see:** the cache grows from about `10 GB` at BF16 to about `14 GB` at FP8 to about `16 GB` at W4A16, and the concurrent 2048-token sequences climb with it, roughly `33`, `46`, and `53`. FP8 alone buys about 13 more concurrent requests on the same card, for almost no accuracy cost. That is the headroom the gauge in section 7 confirms under load.

## 5. Serve the quantized model

vLLM loads compressed-tensors directly. You point the server at the saved directory instead of the original model id. From the JupyterLab terminal you would run:

```bash
vllm serve ./quantized-fp8 --quantization compressed-tensors --port 8001
```

Or, in the Kubernetes manifest from Module 5, change the `--model` arg to the path of the quantized model mounted into the pod and redeploy. Either way the OpenAI-compatible API is identical, so the client code below does not change. For this self-paced module, the quality check runs against whatever model `VLLM_HOST` currently serves, so you can compare by pointing it first at the full-precision endpoint, then at the quantized one.

## 6. The quality check

Memory savings only matter if the answers hold up. Run a small fixed set of prompts and look at the outputs. For a quick numeric signal, use a handful of short questions with known answers and count how many the model gets right. This is not a full benchmark, Module 8 does that, but it catches a quantization scheme that broke the model.

In [ ]:
# Requires a live vLLM endpoint.
# Run a tiny fixed quality check against the currently served model.
from common.config import build_client

client = build_client(settings)

checks = [
    ("What is the capital of France?", "paris"),
    ("What is 12 multiplied by 12?", "144"),
    ("What color is a clear daytime sky?", "blue"),
    ("How many sides does a triangle have?", "3"),
    ("What is the chemical symbol for water?", "h2o"),
]

correct = 0
for question, expected in checks:
    resp = client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": question}],
        max_tokens=32, temperature=0.0,
    )
    answer = resp.choices[0].message.content.strip().lower()
    hit = expected in answer.replace(" ", "")
    correct += hit
    print(f"[{'ok ' if hit else 'miss'}] {question} -> {answer[:50]}")

print(f"\nscore: {correct}/{len(checks)}")

**What you should see:** a per-question result and a score out of five. Run it against the full-precision endpoint, note the score, then point `VLLM_HOST` at the quantized endpoint and run it again. For FP8 the scores should match on a check this simple. The accuracy cost of quantization shows up on harder tasks, which is why Module 8 uses a real evaluation set.

## 7. Confirm the headroom you bought

The reason to do any of this is concurrency. Serving the quantized model at the same `--gpu-memory-utilization` leaves more memory for the KV cache, so `kv_cache_usage_perc` rises more slowly under the same load. Read the gauge against the quantized endpoint and compare it to what you saw in Module 4 at the same concurrency.

In [ ]:
# Requires a live vLLM endpoint (pointed at the quantized model).
# Read the KV cache gauge so you can compare headroom against the full model.
from common import metrics

snap = metrics.snapshot(settings.metrics_url)
print(f"KV cache usage now: {snap['vllm:gpu_cache_usage_perc']*100:.1f}%")
print("Re-run the Module 4 sweep against this endpoint to see the knee move right.")

**What you should see:** the current KV cache usage. The real comparison is the full sweep from Module 4 against the quantized endpoint: with the model taking less memory, the cache fills later, preemption starts later, and the throughput knee moves to a higher concurrency. That is the headroom quantization bought you.

## Things to know

- **The gauge was renamed.** vLLM's V1 engine renamed the KV cache gauge from `vllm:gpu_cache_usage_perc` to `vllm:kv_cache_usage_perc`. The workshop's `snapshot()` returns the value under both names, so the read above works whichever your server exposes. In prose we call it `kv_cache_usage_perc`.
- **FP8 is the safe default.** It roughly halves the weights with little accuracy loss and no calibration step. Reach past it only when memory is the hard constraint.
- **W4A16 is the heavier lever.** It needs a calibration pass over sample text and costs more accuracy, in exchange for shrinking the model to about a quarter of BF16. Lean on Module 8's evaluation before you ship it.
- **`lm_head` stays full precision.** Quantizing the output projection tends to hurt quality for little memory gain, so the recipe ignores it.
- **What is safe to quantize, and what is not.** Risk rises from weights to activations to the KV cache to attention. Weights are large and their errors average out, so they quantize cleanly. Attention repeats across thousands of decode steps, so small errors compound: truncating Pi from 3.14159 to 3 is a 4.5 percent error, but cubed it is 12.9 percent. That is why the recipe quantizes `Linear` weights, keeps `lm_head` in full precision, and why quantizing the KV cache wants an eval pass before you ship it.
- **FP8 over INT8, for the dynamic range.** FP8 E4M3 spends 4 of its 8 bits on an exponent, so it holds the occasional large activation that INT8's evenly-spaced range would clip. That range is why FP8 is the safe default with no calibration step, while INT8 needs per-channel scales to stay accurate.
- **Disk size is a proxy, not the GPU number.** The on-disk footprint tells you roughly how much weight memory you freed; the gauge under load tells you what that bought in KV cache headroom.

> NOTE: The conversion needs a CUDA GPU with room for the unquantized model. If `from_pretrained(..., device_map="cuda")` runs out of memory, you are on too small a card for this source model; pick a smaller one or run the step on a larger GPU.

## Try it yourself

**Run the quality check on both endpoints.** Point `VLLM_HOST` at the full-precision model, record the score, then point it at the quantized model and compare. For FP8 on this simple set the scores should match. **Stretch:** add three harder questions of your own and see whether the quantized model slips first.

**Confirm the headroom under load.** Re-run the Module 4 sweep against the quantized endpoint and watch where `kv_cache_usage_perc` crosses the line where preemption began. The knee should sit at a higher concurrency than the full-precision model.

**Try W4A16.** Change the scheme in the recipe to `"W4A16"` and pass a small calibration dataset to `oneshot`. Measure the new footprint and re-run the quality check; expect a larger drop in both size and accuracy.

In [ ]:
# Change the source model, then run the cell.
ALT_SOURCE = SOURCE_MODEL        # try a different base model id
ALT_QUANT_DIR = os.path.join(os.getcwd(), "quantized-alt")
print("would quantize:", ALT_SOURCE)
print("would write to:", ALT_QUANT_DIR)
# To run the real conversion, copy the recipe + oneshot cell above and
# swap SOURCE_MODEL for ALT_SOURCE and QUANT_DIR for ALT_QUANT_DIR.

## Summary

- Tuning config runs out when the model fills the card. Quantization shrinks the model itself, so it is the next lever after Module 5.
- FP8 with a single LLM Compressor `oneshot` pass roughly halves the weight memory with little accuracy loss and no calibration.
- The on-disk footprint near half BF16 is a proxy for the GPU memory you freed, and at the same `--gpu-memory-utilization` that memory becomes KV cache.
- A bigger cache holds more concurrent requests, so `kv_cache_usage_perc` rises slower and the throughput knee from Module 4 moves right. That is the headroom you bought.

## Next

**Module 7: Two Models, One GPU.** With a smaller footprint per model, you can fit more than one on a single card. Next you serve two models on one GPU, route between a fast one and a smart one, and watch them contend for the same KV cache.